In [ ]:
# hide
import numpy as np
import pyquist as pq
from scipy.signal import lfilter

F_S = 44100


# Two filter designs from the Audio EQ Cookbook (see code/rbj.py). Each returns
# the feedforward (b) and feedback (a) coefficients of a resonant biquad.
def lpf(f_c, Q, f_s):
    w0 = 2 * np.pi * f_c / f_s
    c, alpha = np.cos(w0), np.sin(w0) / (2 * Q)
    b = np.array([(1 - c) / 2, 1 - c, (1 - c) / 2])
    a = np.array([1 + alpha, -2 * c, 1 - alpha])
    return b / a[0], a / a[0]


def hpf(f_c, Q, f_s):
    w0 = 2 * np.pi * f_c / f_s
    c, alpha = np.cos(w0), np.sin(w0) / (2 * Q)
    b = np.array([(1 + c) / 2, -(1 + c), (1 + c) / 2])
    a = np.array([1 + alpha, -2 * c, 1 - alpha])
    return b / a[0], a / a[0]

In [ ]:
# Subtractive synthesis: carve two percussion sounds out of white noise, then
# arrange them into a rhythm with a Score.

def noise_lo(duration, **kwargs):
    """A soft "thump": low-passed noise with a fast decay."""
    n = int(duration * F_S)
    x = np.random.uniform(-1, 1, n)
    b, a = lpf(f_c=180, Q=1.0, f_s=F_S)
    y = lfilter(b, a, x)
    env = np.exp(-np.linspace(0, 10, n))
    return pq.Audio((0.9 * y * env).astype(np.float32), F_S)


def noise_hi(duration, **kwargs):
    """A crisp "tick": high-passed noise with a very fast decay."""
    n = int(duration * F_S)
    x = np.random.uniform(-1, 1, n)
    b, a = hpf(f_c=6000, Q=0.8, f_s=F_S)
    y = lfilter(b, a, x)
    env = np.exp(-np.linspace(0, 45, n))
    return pq.Audio((0.9 * y * env).astype(np.float32), F_S)


def drum(voice, duration, **kwargs):
    """Dispatch each event to the right voice."""
    return noise_lo(duration) if voice == "lo" else noise_hi(duration)


# A 16-step pattern: low "thump" on the strong beats, high "tick" on every step.
beat = 0.22
kicks = {0, 4, 8, 12}
events = []
for step in range(16):
    events.append((step * beat, {"voice": "hi", "duration": 0.12}))
    if step in kicks:
        events.append((step * beat, {"voice": "lo", "duration": 0.35}))

rhythm = pq.Score(events)
pq.play(rhythm.render(drum))